# Wasserstein-2 Distance with Swiss Roll

This notebook demonstrates translating and rotating a swiss roll density and calculating the Wasserstein-2 distance.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from monge_ampere.optimal_transport import solve_ot, wasserstein2
import monge_ampere.boundary as bc
import warnings

warnings.filterwarnings("ignore")

## Define Swiss Roll Density

We generate a periodic swiss roll density on a grid. To ensure numerical stability, we add a small background density.


In [ ]:
def make_swiss_roll(n, center=(0.5, 0.5), rotation=0.0, translation=(0.0, 0.0), sigma=0.03):
  """
  Generate a periodic swiss roll density on the grid.
  
  Args:
    n: Grid size
    center: Base center of the spiral
    rotation: Rotation angle in radians
    translation: Translation vector
    sigma: Gaussian blur width
  Returns:
    Normalized density array
  """
  h = 1.0 / n
  x = np.arange(n) * h
  X, Y = np.meshgrid(x, x, indexing="ij")
  
  # Generate spiral points
  t = np.linspace(0.5, 3.0, 100) * np.pi
  r = t / (4.0 * np.pi)
  
  sx = r * np.cos(t)
  sy = r * np.sin(t)
  
  # Rotate
  cos_rot = np.cos(rotation)
  sin_rot = np.sin(rotation)
  rx = cos_rot * sx - sin_rot * sy
  ry = sin_rot * sx + cos_rot * sy
  
  # Translate
  cx, cy = center
  tx, ty = translation
  
  px = rx + cx + tx
  py = ry + cy + ty
  
  # Baseline density
  rho = np.ones_like(X) * 1e-3
  
  # Sum Gaussians over periodic domain
  for i in range(len(t)):
    for di in [-1, 0, 1]:
      for dj in [-1, 0, 1]:
        dx = X - px[i] - di
        dy = Y - py[i] - dj
        rho += np.exp(-(dx**2 + dy**2) / (2.0 * sigma**2))
        
  rho /= (np.sum(rho) * h * h)
  return rho

## Create Source and Target Densities

Here we create a source swiss roll and a target which is translated and rotated.


In [ ]:
n = 64
h = 1.0 / n

# Source: baseline swiss roll
rho0 = make_swiss_roll(n, center=(0.5, 0.5), rotation=0.0, translation=(0.0, 0.0), sigma=0.02)

# Target: translated and rotated swiss roll
# We translate by (0.2, 0.1) and rotate by pi/4
trans = (0.2, 0.1)
rot = np.pi / 4.0
rho1 = make_swiss_roll(n, center=(0.5, 0.5), rotation=rot, translation=trans, sigma=0.02)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(rho0.T, origin="lower", extent=[0, 1, 0, 1], cmap="Blues")
axes[0].set_title(r"Source Density $\rho_0$")
axes[1].imshow(rho1.T, origin="lower", extent=[0, 1, 0, 1], cmap="Oranges")
axes[1].set_title(r"Target Density $ho_1$")
plt.show()

## Solve Optimal Transport

We solve for the Wasserstein-2 distance and print the calculated and expected distances.


In [ ]:
# Solve Monge-Ampère
result = solve_ot(
  rho0, 
  rho1, 
  h, 
  solver="newton", 
  dw=1, 
  max_iter=30, 
  ot_max_iter=5, 
  damping=0.5
)

w2 = np.sqrt(result.w2_squared)
print(f"Calculated W2 Distance: {w2:.5f}")

# The expected distance is bounded from below by the translation magnitude
# due to the rotation and shape changes
d_trans = np.sqrt(trans[0]**2 + trans[1]**2)
print(f"Translation Magnitude: {d_trans:.5f}")

## Visualize the Optimal Transport Map

We plot the periodic perturbation potential and the displacement field.


In [ ]:
x_mesh = np.arange(n) * h
X_mesh, Y_mesh = np.meshgrid(x_mesh, x_mesh, indexing="ij")
Tx, Ty = result.transport_map
disp_x = Tx - X_mesh
disp_y = Ty - Y_mesh

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(result.psi.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
axes[0].set_title(r"Periodic Perturbation Potential $\psi$")
plt.colorbar(im0, ax=axes[0])

step = 3
axes[1].quiver(
  X_mesh[::step, ::step], Y_mesh[::step, ::step], 
  disp_x[::step, ::step], disp_y[::step, ::step],
  angles='xy', scale_units='xy', scale=1.0, color='r', alpha=0.8
)
axes[1].imshow(rho0.T, origin="lower", extent=[0, 1, 0, 1], cmap="Blues", alpha=0.5)
axes[1].set_title(r"Displacement Field $T(x) - x$ over $\rho_0$")
plt.show()